# B2B Customer Churn — Feature Engineering & Preprocessing

**Purpose:** Transform raw account data into an ML-ready feature matrix for XGBoost churn modeling.

**Outputs (saved with joblib):**
- Fitted `ColumnTransformer` preprocessing pipeline
- Stratified train/test feature matrices and labels

**Note:** No model training in this notebook — only data preparation.


In [ ]:
# --- Imports ---
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Paths
DATA_PATH = Path("../data/customer_churn_business_dataset.csv")
ARTIFACTS_DIR = Path("../models/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "churn"
ID_COLS = ["customer_id"]

# Columns for IQR outlier capping (must exist in dataset)
IQR_COLS = [
    "monthly_logins",
    "avg_session_time",
    "total_revenue",
    "support_tickets",
    "payment_failures",
]

pd.set_option("display.max_columns", None)


## 1. Load data

We start from the business churn export and work on a copy so the raw load remains unchanged for audit.


In [ ]:
df = pd.read_csv(DATA_PATH)
df_clean = df.copy()

print(f"Loaded {len(df_clean):,} rows × {df_clean.shape[1]} columns")
df_clean.head()


## 2. Missing value treatment

| Type | Strategy | Rationale |
|------|----------|-----------|
| Numerical | Median imputation | Robust to skewed B2B metrics |
| Categorical | Mode imputation | Preserves most frequent segment/contract |

Median/mode statistics are computed on the cleaning frame; in production, persist these from **training data only** to avoid leakage.


In [ ]:
missing_before = df_clean.isnull().sum()
missing_before[missing_before > 0]


In [ ]:
NUM_COLS_RAW = df_clean.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore").tolist()
CAT_COLS_RAW = [c for c in df_clean.columns if c not in NUM_COLS_RAW + ID_COLS + [TARGET]]

# Impute numerical features (exclude target)
num_impute_cols = [c for c in NUM_COLS_RAW if c != TARGET]
for col in num_impute_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Impute categorical features
for col in CAT_COLS_RAW:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0])

remaining = df_clean.isnull().sum().sum()
print(f"Missing values after imputation: {remaining}")


## 3. Duplicate removal

Duplicate rows (or duplicate `customer_id`) would inflate metrics and distort stratified sampling.


In [ ]:
dup_rows = df_clean.duplicated().sum()
dup_ids = df_clean.duplicated(subset=ID_COLS).sum()

rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=ID_COLS, keep="first")

print(f"Fully duplicate rows removed: {dup_rows}")
print(f"Duplicate IDs removed: {dup_ids}")
print(f"Rows before: {rows_before:,} → after: {len(df_clean):,}")


## 4. Outlier handling (IQR method)

For heavy-tailed usage and billing fields we **winsorize** to the Tukey fences (Q1 − 1.5×IQR, Q3 + 1.5×IQR) instead of dropping rows, preserving sample size for rare enterprise accounts.


In [ ]:
def iqr_cap(series: pd.Series) -> pd.Series:
    """Clip series to IQR-based lower/upper bounds."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)


outlier_report = []
for col in IQR_COLS:
    if col not in df_clean.columns:
        raise KeyError(f"Required column missing for IQR: {col}")
    q1, q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (df_clean[col] < lower) | (df_clean[col] > upper)
    outlier_report.append({"column": col, "outliers": int(mask.sum()), "pct": round(mask.mean() * 100, 2)})
    df_clean[col] = iqr_cap(df_clean[col])

pd.DataFrame(outlier_report)


## 5. Feature typing

Separate numeric and categorical predictors before encoding and composite feature construction.


In [ ]:
# Refresh column lists after cleaning
NUM_COLS = df_clean.select_dtypes(include=[np.number]).columns.drop(TARGET, errors="ignore").tolist()
CAT_COLS = [c for c in df_clean.columns if c not in NUM_COLS + ID_COLS + [TARGET]]

print("Numerical columns:", NUM_COLS)
print("Categorical columns:", CAT_COLS)


## 6. Advanced business features

Composite scores capture **multi-signal risk and value** in forms tree models exploit efficiently.

| Feature | Inputs | Interpretation |
|---------|--------|----------------|
| `engagement_score` | logins, active days, session time, features used | Higher → healthier product adoption |
| `support_risk_score` | tickets, resolution time, escalations | Higher → operational friction |
| `inactivity_risk` | days since last login | Higher → disengagement |
| `satisfaction_risk` | CSAT, NPS (inverted) | Higher → dissatisfaction |
| `customer_value_score` | revenue, tenure | Higher → strategic account value |
| `payment_risk_score` | failures, recent price increase | Higher → billing stress |
| `marketing_engagement_score` | email open & click rates | Higher → marketing responsiveness |

Components are scaled with **robust IQR scaling** (median-centered, IQR denominator) for stability under skew.


In [ ]:
def robust_scale(series: pd.Series) -> pd.Series:
    """Median/IQR robust scaling for composite features."""
    med = series.median()
    iqr = series.quantile(0.75) - series.quantile(0.25)
    if iqr == 0:
        return pd.Series(0.0, index=series.index)
    return (series - med) / iqr


# A. Engagement (higher = better engagement)
df_clean["engagement_score"] = (
    robust_scale(df_clean["monthly_logins"])
    + robust_scale(df_clean["weekly_active_days"])
    + robust_scale(df_clean["avg_session_time"])
    + robust_scale(df_clean["features_used"])
) / 4

# B. Support risk (higher = worse)
df_clean["support_risk_score"] = (
    robust_scale(df_clean["support_tickets"])
    + robust_scale(df_clean["avg_resolution_time"])
    + robust_scale(df_clean["escalations"])
) / 3

# C. Inactivity risk
df_clean["inactivity_risk"] = robust_scale(df_clean["last_login_days_ago"])

# D. Satisfaction risk (invert satisfaction metrics)
df_clean["satisfaction_risk"] = (
    -robust_scale(df_clean["csat_score"]) + -robust_scale(df_clean["nps_score"])
) / 2

# E. Customer value (higher = more valuable)
df_clean["customer_value_score"] = (
    robust_scale(df_clean["total_revenue"]) + robust_scale(df_clean["tenure_months"])
) / 2

# F. Payment risk
df_clean["payment_risk_score"] = (
    robust_scale(df_clean["payment_failures"])
    + robust_scale(df_clean["price_increase_last_3m"])
) / 2

# G. Marketing engagement (higher = more engaged)
df_clean["marketing_engagement_score"] = (
    robust_scale(df_clean["email_open_rate"])
    + robust_scale(df_clean["marketing_click_rate"])
) / 2

ENGINEERED_COLS = [
    "engagement_score",
    "support_risk_score",
    "inactivity_risk",
    "satisfaction_risk",
    "customer_value_score",
    "payment_risk_score",
    "marketing_engagement_score",
]

df_clean[ENGINEERED_COLS].describe().round(3)


## 7. Skewness reduction (log transforms)

Right-skewed revenue and usage metrics benefit from `log1p` transforms before scaling. We **retain originals** and add `log_*` columns for the pipeline.


In [ ]:
# Identify highly skewed numeric columns (|skew| > 1), excluding binary/target/engineered
skew_candidates = [
    c for c in NUM_COLS
    if c not in ENGINEERED_COLS + ["price_increase_last_3m", "churn"]
]
skew_series = df_clean[skew_candidates].skew().sort_values(key=abs, ascending=False)
skew_series


In [ ]:
LOG_TRANSFORM_COLS = [
    c for c in skew_series.index if abs(skew_series[c]) > 1 and (df_clean[c] >= 0).all()
]

LOG_COLS = []
for col in LOG_TRANSFORM_COLS:
    log_name = f"log_{col}"
    df_clean[log_name] = np.log1p(df_clean[col])
    LOG_COLS.append(log_name)

print("Log-transformed columns:", LOG_COLS)


## 8. Encoding strategy

- **Nominal categoricals** (`contract_type`, `customer_segment`): `OneHotEncoder` with `drop='first'` to reduce collinearity.
- **Binary numeric flags** (e.g. `price_increase_last_3m`): kept as numeric — no `LabelEncoder` needed.
- **LabelEncoder** is intentionally **not** applied to nominal fields (arbitrary ordinality harms tree splits and linear models).


## 9. Build modeling matrix & stratified split

Hold out a stratified test set **before** fitting the preprocessing pipeline to prevent information leakage from scaling and encoding.


In [ ]:
# Features for modeling (drop identifiers and target)
DROP_FOR_MODEL = ID_COLS + [TARGET]
FEATURE_COLS = [c for c in df_clean.columns if c not in DROP_FOR_MODEL]

X = df_clean[FEATURE_COLS].copy()
y = df_clean[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}")


## 10. Sklearn preprocessing pipeline

`ColumnTransformer` applies:
- `StandardScaler` on numeric columns (raw, log, and engineered)
- `OneHotEncoder` on categorical columns

Fitted **only on training data**, then applied to train and test.


In [ ]:
# Update numeric list after feature engineering
NUM_FOR_PIPELINE = X_train.select_dtypes(include=[np.number]).columns.tolist()
CAT_FOR_PIPELINE = [c for c in X_train.columns if c not in NUM_FOR_PIPELINE]

numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())],
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
        )
    ],
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUM_FOR_PIPELINE),
        ("cat", categorical_transformer, CAT_FOR_PIPELINE),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

churn_preprocessor = Pipeline(steps=[("preprocessor", preprocessor)])

# Fit on train, transform both splits
X_train_processed = churn_preprocessor.fit_transform(X_train)
X_test_processed = churn_preprocessor.transform(X_test)

encoded_feature_names = churn_preprocessor.named_steps["preprocessor"].get_feature_names_out()

print(f"Final feature count: {X_train_processed.shape[1]}")
print(f"Raw input columns: {X_train.shape[1]}")
print("Train processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)


In [ ]:
# Encoded feature names (for SHAP / dashboards later)
pd.Series(encoded_feature_names, name="feature")


## 11. Persist artifacts

Serialized objects support the training notebook, FastAPI inference, and reproducible deployments.


In [ ]:
# Save preprocessing pipeline and split datasets
joblib.dump(churn_preprocessor, ARTIFACTS_DIR / "churn_preprocessor.joblib")
joblib.dump(X_train_processed, ARTIFACTS_DIR / "X_train.joblib")
joblib.dump(X_test_processed, ARTIFACTS_DIR / "X_test.joblib")
joblib.dump(y_train, ARTIFACTS_DIR / "y_train.joblib")
joblib.dump(y_test, ARTIFACTS_DIR / "y_test.joblib")
joblib.dump(encoded_feature_names, ARTIFACTS_DIR / "feature_names.joblib")

saved = sorted(p.name for p in ARTIFACTS_DIR.glob("*.joblib"))
print("Saved artifacts:")
for name in saved:
    print(f"  - models/artifacts/{name}")


## 12. Summary

**Pipeline complete.** Next steps (separate notebook):
1. Train XGBoost on `X_train` / `y_train` with class imbalance handling.
2. Evaluate on `X_test` (precision-recall, ROC-AUC, calibration).
3. Compute SHAP values using `feature_names.joblib` for explainability.
4. Wire the same `churn_preprocessor` in FastAPI for consistent inference.

**Production reminders:**
- Refit imputation, IQR bounds, and `StandardScaler` on rolling training windows.
- Version artifacts (`churn_preprocessor.joblib`) alongside model binaries.
- Apply identical feature engineering functions in batch scoring jobs before `transform()`.
